# Feature Extraction

Bu aşamada ham video/ses/text, modelin okuyabileceği sayı vektörlerine çevirip hdf5 dosyasına yazacağız. Yani bir nevi modelin girdisini hazırlayacağız.

# Model Ne İstiyor ?

İlk olarak model bizden ne istiyor ona bir bakalım. Bunun için TripleSumm modelinin orjinalinde dataset.py dosyasını okuyarak, modelin veriyi nasıl yüklediğine baktık.

Dosyayı okuyunca 2 farklı zaman ekseni bulunmakta. Görüntü, ses ve hizalamadan gelen gt_score aynı eksende, yani saniye ekseninde hizalı olmalı. (N = videonun saniye sayısı). Text ise ayrı bir M ekseninde, yani cümle ekseninde hizalı olmalı.

Ayrıca dosyaya bakarak modelin okudu veya okuyacağı veri isimlerini ve okuma biçimlerini gördük. - hangi key'lerden okuyor, okurken ham mı veriyor gibi. Model feature'ları ham olarak alıyor, okuma sırasında ekstra bir dönüşüm uygulamıyor (sadece batch içi padding var). Yani feature'ı diske hangi biçimde yazarsak, modele aynen o biçimde gidiyor.

# N = int(toplam_kare / fps)

Yukarıda bahsettiğimiz gibi visual, audio ve gt_score (0-1 etiketli vektör) N yani saniye ekseninde hizalı olmak zorunda. Yani videonun her saniyesi encoder'dan geçtikten sonra elde ettiğimiz vektörün boyutu[N, 2048] olmalı.

"Aynı N ekseninde hizalı" derken kastedilen şu: üçünün de satır sayısı aynı N, ve her üçünde de i. satır aynı saniyeye ait. Yani visual[i] (i. saniyenin görüntü vektörü), audio[i] (i. saniyenin ses vektörü) ve gt_score[i] (i. saniyenin önem skoru) hepsi aynı fiziksel saniyeyi anlatıyor.

Peki bu üç şey farklı kaynaklardan, farklı araçlarla üretiliyor: visual videodan (kare sayarak), audio ses dosyasından (saniye dilimleyerek), gt_score ise hizalama pipeline'ından. Her biri "videonun uzunluğu kaç saniye" sorusuna kendi yöntemiyle cevap verirse, bu cevaplar birbirini tutmayabilir. Videonun uzunluğu" göründüğü kadar kesin bir sayı değil. Bir video 6076.0 saniye sürebilir, ama ses dosyası çıkarıldığında 6075.99 saniye ölçülebilir (kodlama/dilimleme artıkları yüzünden). Kare sayısını fps'e bölünce tam sayı çıkmayabilir. Her araç bu ondalık artığı farklı yuvarlarsa, farklı N'ler doğar. Yani sorun büyük bir hata değil, saniyenin küçük kesirlerinden doğan, kolayca gözden kaçan bir kayma.

Bu kaymayı ortadan kaldırmanın yolu, N'i her modalitenin kendi başına hesaplamasına izin vermemek; tek bir otoriter tanım koyup herkesi ona uydurmak. Seçtiğimiz tanım:

$N = int(toplam kare / fps)$

Videoyu cv2 ile açıp toplam kare sayısını fps'e böler, tam sayıya (aşağı) yuvarlarız. Bu N, "bu videonun kaç saniyelik feature üreteceği"nin tek kaynağı olur.

Yani kısaca her encoder için bir N üretmektense N'yi tek bir otoriter formülle üretip, karışıklığı önledik.



# Visual - InceptionV3

InceptionV3 ile her saniyenin görüntüsünü modelin işleyebileceği bir sayı vektörüne dönüştüreceğiz. InceptionV3 ImageNet üzerinde milyonlarca görüntüyle eğitilmiş; bir kareye baktığında "bu görüntüde ne var" bilgisini yoğun bir vektöre sıkıştırmayı çoktan öğrenmiş durumda. Bu yüzden onu sıfırdan eğitmek yerine frozen tutup kendi içeriğimizi ondan geçiriyoruz.

Tek fark sınıflandırıcı olarak tasarlanmış bu encoderi, son layer'ini atarak farklı bir şekilde kullanıyoruz.

Bu encoderin cıktısı 2048 boyutlu bir özet vektörü üretir. Bu vektör görüntüde ne varı okunabilir bir etiket olarak değil, ayırt edici bir koordinat olarak taşır.

## Preprocessing

Frameleri doğrudan encodera veremeyiz. Çünkü encoder önceden eğitilirken belirli girdi biçimleriyle üzerinden eğitildi. Bunlar:

- belirli bir boyut
- belirli bir renk sıralaması
- belirli bir değer aralığı
- normalizasyon.

biz de kendi frame'lerimizi bu işlemlerden geçirdik.





# Audio - AST (audio spectrogram transformer)

Ses kanalının görevi de her saniye için bir vektör üretmek, N eksenine hizalı bir şekilde. Ama sesin örnekleme mantığı ile visual örnekleme mantığı çok farklı.

16khz 'e indirme - Kullandığımız encoder AST, AudioSet üzeridne 16 khZ örnekleme ile eğitildi. Bu yüzden bizim input audio da önce 16 kHz'e resample edilir.

Peki sesi saniye saniye mi işleyeceğiz? Hayır, 10 saniyelik ses penceleri ile işleriz. Çünkü Visual'de her saniye için o saniyenin tek karesini aldık, çünkü bir görüntü tek başına anlamlı. Ama ses için tek bir saniye çoğu zaman anlamsızdır.Bu yüzden her saniye için, o saniyenin etrafındaki 10 saniyeyi encoder'a veriyoruz: merkezi [t−5, t+5] penceresi. Yani i. saniyenin vektörü, i-5'ten i+5'e kadarki sesi "duyarak" üretiliyor.


Bu, ardışık saniyelerin pencerelerinin örtüşmesi demek: i. saniye ile i+1. saniyenin pencereleri 9 saniyeyi ortak görür. Bu örtüşme kasıtlı — komşu saniyeler arasında yumuşak, bağlamlı bir geçiş sağlıyor. Kenar durumlar (videonun ilk 5 ve son 5 saniyesi, pencerenin dışarı taştığı yerler) zero-padding ile dolduruluyor.


## CLS Token

AST gibi transformer tabanlı modeller, girdinin başına özel bir CLS (classification) token ekler. Bu token bir ses parçasına karşılık gelmez — özellikle "tüm girdinin özetini topla" görevi için konulmuş bir yer tutucudur. Model çalışırken bu token, attention mekanizması aracılığıyla diğer bütün patch token'larından bilgi çeker; sonuçta CLS token'ının çıktısı, tüm 10 saniyelik pencerenin bütüncül bir özeti hâline gelir.


AST ise bize tek bir vektör değil, çok sayıda vektör döndürür. Sesin spektrogramını küçük parçalara (patch) bölüp her parça için ayrı bir temsil üretir — yüzlerce token. Bizim istediğimiz ise o saniyenin tek bir 768-d özetidir.

Bizde o özeti elde etmek için direkt olarak CLS tokenlarını kullanıyoruz. Çünkü CLS, tam da bu özetleme işi için eğitilmiş özel bir temsil; modelin kendisi eğitim boyunca "tüm girdiyi bu token'da topla" diye öğrenmiş. Diğer yüzlerce tokenin ortalamasını almak, cls yanında biraz kullanışsız kalır.


# Text - BERTurk

Text modalitesi diğerlerinden biraz farklı durumda. önce neden zaman eksini (N) yerine cümle ekseni (M) ona bakalım.

Visual ve audio saniye ekseninde (N) çalışıyordu — her saniye bir vektör. Text bu kalıba uymuyor, çünkü metnin doğal birimi saniye değil, cümle. Metni saniyeye bölmek anlamı parçalar. Bu yüzden text ayrı bir eksende, M (cümle) ekseninde yaşıyor: M o videodaki cümle sayısı. Bu sebeple bu modaliteyi cümle cümle işleme alacağız.

## Whisper

Metin üretme işlemini Whisper ile konuşmayı metne çevirerek yapacağız.  Whisper (faster-whisper, large-v3) kullanıyoruz. Whisper sadece "ne söylendi"yi değil, ne zaman söylendiği bilgisini de veriyor — her cümle için başlangıç/bitiş zaman damgası. Yani çıktı: zaman damgalı segmentler (cümleler). Bu zaman damgaları kritik, çünkü Parça 2'de cümleyi doğru saniyeye yerleştirmek için onları kullanacağız.

## BERTurk

BERTurk encoder ile de bu elde ettiğimiz cümleleri 768 boyutlu bir vektöre çeviriyoruz.


ses → Whisper (zaman damgalı cümleler) → BERTurk → [M, 768]



Paperin RoBERTa'sı yerine Türkçe için BERTurk kullanıldı.

# Broadcast

Text encoder'ından sonra elimizde [M, 768]'lik bir vektör var.(M cümle, her biri bir 768-d vektör)
Fakat model bizden her modalitenin N yani saniye ekseninde olmasını bekliyor ki hizalama yapabilsin. Bu sebeple cümle-uzayındaki matrisi saniye-uzayına çevireceğiz. (broadcasting yapacağız)

Peki nasıl ?
Whisper'ın bize erdiği zaman damgaları ile herr cümlenin hangi saniyeler arasında söylendiğini biliyoruz. Yani, bir cümle 67. ile 71. saniyeler arasında söylendiyse, o cümlenin 768-d vektörünü 67, 68, 69, 70, 71. saniyelerin hepsine kopyalarız. Yani cümlenin tek vektörü, kapsadığı tüm saniyelere yayılır ("broadcast" edilir). Sonuç olarak [N, 768] boyutlu bir matris olustururuz.

Böylece text de visual/audio ile aynı N ekseninde, i. satır aynı saniyeyi anlatıyor.


## < pad > tokeni
Burada kritik konu < pad > tokeni. Maçta konuşma olmayan yerleri 0 vektörü koymak yerine < pad > tokeni koyarız. Çünkü Sıfır vektörü, modele "burada güçlü ama nötr bir sinyal var" gibi görünebilir — sıfır, model için anlamlı bir değerdir, "yokluk" değil. Bunun yerine, konuşmasız saniyeler < pad > token'ının default vektörüyle dolduruluyor. < pad >, BERTurk'ün "burada içerik yok" anlamına gelen özel token'ı; onun vektörü, modele "bu saniyede metin sinyali yok" bilgisini doğru biçimde iletir. Yani boşluğu sıfırla değil, "boşluk"un kendi anlamlı temsiliyle dolduruyoruz.

# Paketleme

Her modalite başına bir dosya açıp, içerisine maçlar grup olacak şekilde dosyalıyoruz.

```pipelines/
│
├── trt_feat_visual_inceptionv3.h5
│   ├── fcsb-fenerbahce/
│   │   └── features        (5917, 2048)  float32
│   └── liverpool-realmadrid/
│       └── features        (6076, 2048)  float32
│
├── trt_feat_audio_ast.h5
│   ├── fcsb-fenerbahce/
│   │   └── features        (5917, 768)   float32
│   └── liverpool-realmadrid/
│       └── features        (6076, 768)   float32
│
├── trt_feat_text_berturk.h5            ← broadcast CLS, [N,768]
│   ├── fcsb-fenerbahce/
│   │   └── features        (5917, 768)   float32
│   └── liverpool-realmadrid/
│       └── features        (6076, 768)   float32
│
├── trt_gt.h5
│   ├── fcsb-fenerbahce/
│   │   ├── gt_score        (5917,)   float32   ← MSE hedefi (füzyon, gaussian)
│   │   ├── gt_summary      (5917,)   int64     ← ikili birleşim, rezerve (F1)
│   │   └── change_points   (1, 2)    int64     ← placeholder [[0, 5916]]
│   └── liverpool-realmadrid/
│       ├── gt_score        (6076,)   float32
│       ├── gt_summary      (6076,)   int64
│       └── change_points   (1, 2)    int64     ← [[0, 6075]]
│
└── trt_split.json
        { "train": ["liverpool-realmadrid"],
          "val":   ["fcsb-fenerbahce"],
          "test":  ["fcsb-fenerbahce"] }        ← smoke-test; gerçek bölünme 40-50 maçta

yan dosyalar (esneklik yedeği — model okumaz):
├── feats_text_cls_{mac}.npy     ham [M, 768]   (broadcast öncesi, cümle bazlı CLS)
├── feats_text_mean_{mac}.npy    ham [M, 768]   (mean pooling, ablation için)
├── transcript_{mac}.json        Whisper zaman damgaları + cümleler
└── (visual/audio için de AKTIF_MAC adlı .npy ara-kayıtlar) ```